In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression,ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,root_mean_squared_error,confusion_matrix,roc_auc_score,log_loss
from sklearn.metrics import f1_score,accuracy_score,recall_score,precision_score,classification_report
from tqdm import tqdm
import os
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler,PolynomialFeatures
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
import datetime as dt

In [ ]:
train = pd.read_csv("train.csv")
train

In [ ]:
le=LabelEncoder() 
train['Churn']=le.fit_transform(train['Churn'])
train['Churn']

In [ ]:
X = train.drop('Churn', axis=1)
y = train['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state=26, stratify= train['Churn'])

In [ ]:
from sklearn.compose import make_column_selector,ColumnTransformer
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")
trans = ColumnTransformer(
    transformers=[("OHE", ohe,make_column_selector(dtype_include=object))],remainder="passthrough",
    verbose_feature_names_out=False).set_output(transform="pandas")

In [ ]:

X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)

scalar=StandardScaler().set_output(transform="pandas")
scalar.fit(X_trn_ohe)
X_trn_scl = scalar.transform(X_trn_ohe)
X_tst_scl = scalar.transform(X_tst_ohe)

In [27]:
ks=[1,2,3,4,5,6,7,8,9,10]
scores = []
for k in tqdm(ks) :
    knn= KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_trn_scl, y_train)
    y_pred_prob = knn.predict_proba(X_tst_scl)
    scores.append([k, log_loss(y_test, y_pred_prob)])
df_scores = pd.DataFrame(scores, columns=['ks', 'log_loss'])
df_scores.sort_values('log_loss', ascending=True)

In [26]:
test = pd.read_csv("test.csv")
test

In [30]:
X_ohe = trans.fit_transform(X)
X_scalar=scalar.fit_transform(X_ohe)
bm = KNeighborsClassifier(n_neighbors=10, n_jobs=-1)
bm.fit(X_scalar,y)

In [31]:
test = pd.read_csv("test.csv") # Remove index_col=0
test_ohe = trans.transform(test)
test_scl = scalar.transform(test_ohe)

In [32]:
submit = pd.read_csv('Sample_Submission.csv')
# Use the scaled test data here
y_pred_prob = bm.predict_proba(test_scl) 
submit["churn"] = y_pred_prob[:, 1]

In [ ]:
submit

In [ ]:
submit.to_csv("sbt_knn_20_Apr.csv",index=False)
submit[submit["churn"]>=0.5]